# Existing-source forward raw price plane

Mechanical split-basis repair only. No strategy result.

In [1]:
import hashlib,json,os
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
import numpy as np,pandas as pd
from qlib.workflow import R
from src.data.qlib_native import build_qlib_store_from_frame,QlibProviderConfig,load_qlib_feature_frame
from src.core.recorder_sources import load_verified_recorder_objects
root=Path.cwd();card=root/'research/soxx_forward_raw_split_plane_20261112';plan=json.loads((card/'plan.json').read_text());src=plan['sources']
if (card/'summary.json').exists():raise ValueError('fixed derived plane already sealed; do not overwrite')
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
for key,digest in [('soxl_futu','soxl_sha256'),('soxx_futu','soxx_sha256'),('prior_anchor_panel','prior_anchor_sha256')]:
 assert sha(root/src[key])==src[digest]
anchor=pd.read_csv(root/src['prior_anchor_panel']).set_index('date').loc['2026-10-08']
raw={sym:pd.read_csv(root/src[sym.lower()+'_futu']).set_index('date').loc['2026-10-09':'2026-11-11'].copy() for sym in ['SOXL','SOXX']}
assert len(raw['SOXL'])==len(raw['SOXX'])==24
assert raw['SOXL'].index.equals(raw['SOXX'].index) and not raw['SOXX'].index.has_duplicates
split_day='2026-11-05';presplit_adjusted=['2026-10-30','2026-11-02','2026-11-03','2026-11-04']
rows=[];checks=[]
for sym in ['SOXL','SOXX']:
 data=raw[sym];prior_raw=float(anchor[sym.lower()+'_close'])
 for day,r in data.iterrows():
  factor=3 if sym=='SOXX' and day in presplit_adjusted else 1
  split_factor=3 if sym=='SOXX' and day==split_day else 1
  vendor_prior=float(r.last_close)*factor
  comparable_prior=prior_raw/split_factor
  prev_error=abs(vendor_prior-comparable_prior)
  if prev_error>1e-5:raise ValueError(f'{sym} vendor previous close does not match raw-share basis on {day}: {prev_error}')
  values={f:float(r[f])*factor for f in ['open','high','low','close']}
  if not all(np.isfinite(list(values.values()))) or min(values.values())<=0 or values['low']>min(values['open'],values['close']) or values['high']<max(values['open'],values['close']):
   raise ValueError(f'{sym} raw OHLC invalid on {day}')
  actual=split_factor*values['close']/prior_raw-1
  vendor=float(r.change_rate)/100
  error=abs(actual-vendor)
  if error>1e-6:raise ValueError(f'{sym} return/change_rate mismatch on {day}: {error}')
  rows.append({'instrument':sym,'datetime':day,**values})
  checks.append({'instrument':sym,'date':day,'price_multiplier':factor,'split_shares_after_open':split_factor,'prior_raw_close':prior_raw,'vendor_last_close':float(r.last_close),'derived_raw_close':values['close'],'actual_return':actual,'vendor_change_rate':vendor,'prior_error':prev_error,'return_error':error})
  prior_raw=values['close']
# The anchor is a price identity, not a new trade or strategy return.
for sym in ['SOXL','SOXX']:
 rows.append({'instrument':sym,'datetime':'2026-10-08','open':float(anchor[sym.lower()+'_open']),'high':float(anchor[sym.lower()+'_high']),'low':float(anchor[sym.lower()+'_low']),'close':float(anchor[sym.lower()+'_close'])})
frame=pd.DataFrame(rows).sort_values(['datetime','instrument']).reset_index(drop=True);check=pd.DataFrame(checks)
assert len(frame)==50 and len(check)==48
provider=root/plan['output_provider']
manifest=build_qlib_store_from_frame(frame,provider_uri=provider,freq='day',as_of='2026-11-11',source='Existing retained Futu OHLC plus official 2026-11-05 SOXX split, no volume')
read=load_qlib_feature_frame(QlibProviderConfig(provider,freq='day'),instruments=['SOXL','SOXX'],fields=['$open','$high','$low','$close'],start='2026-10-08',end='2026-11-11')
for row in frame.itertuples(index=False):
 for f in ['open','high','low','close']:
  observed=float(read.loc[(row.instrument,pd.Timestamp(row.datetime)),'$'+f]);expected=float(getattr(row,f))
  if abs(observed-expected)>5e-5:raise ValueError(f'Qlib raw readback differs {row.instrument} {row.datetime} {f}')
report={'status':'price_only_forward_raw_plane_qualified_under_split_source','historical_research_passed':False,'soxl_source_sha256':src['soxl_sha256'],'soxx_source_sha256':src['soxx_sha256'],'anchor_panel_sha256':src['prior_anchor_sha256'],'provider_manifest_sha256':sha(provider/'qlib_store_manifest.json'),'rows':len(frame),'source_sessions':24,'split_day':split_day,'split_ratio':3,'pre_split_adjustment_days':presplit_adjusted,'max_prior_close_error':float(check.prior_error.max()),'max_vendor_return_error':float(check.return_error.max()),'qlib_ohlc_error_bound':5e-5,'new_fits':0,'new_candidate_accounts':0,'source_scope':'derived existing-source price only; no certified physical volume, complete post-Oct9 dividends, first receipt, broker execution or strategy forward pass'}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 R.save_objects(frozen_plan=plan,source_identity=src,derived_raw_prices=frame,price_checks=check,provider_manifest=manifest,report=report)
 rec=R.get_recorder();rid,eid=rec.id,str(rec.experiment_id)
local=card/'qlib_recorder'/eid/rid;names=['frozen_plan','source_identity','derived_raw_prices','price_checks','provider_manifest','report']
verification={'recorder_id':rid,'entries':[{'path':(local/'artifacts'/k).relative_to(card).as_posix(),'sha256':sha(local/'artifacts'/k),'bytes':(local/'artifacts'/k).stat().st_size} for k in names]}
(card/'verification_manifest.json').write_text(json.dumps(verification,indent=2)+'\n')
summary={'experiment':plan['experiment'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str((card/'qlib_recorder').resolve()),'recorder_local_dir':str(local.resolve()),'plan_sha256':sha(card/'plan.json'),**report,'artifact_policy':{'policy':'slim_card_recorder_first','recorder_only_artifacts':names}}
(card/'summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False)+'\n')
objects,rr=load_verified_recorder_objects(card,expected_recorder_id=rid,artifacts=['derived_raw_prices','price_checks','report'],expected_manifest_sha256=sha(card/'verification_manifest.json'))
pd.testing.assert_frame_equal(objects['derived_raw_prices'],frame);pd.testing.assert_frame_equal(objects['price_checks'],check);assert objects['report']==report
(card/'readback_receipt.json').write_text(json.dumps({'recorder_id':rid,'summary_sha256':sha(card/'summary.json'),'manifest_sha256':sha(card/'verification_manifest.json'),'readback':rr},indent=2)+'\n')
print(json.dumps({'status':report['status'],'rows':len(frame),'max_prior_error':report['max_prior_close_error'],'max_return_error':report['max_vendor_return_error'],'provider_manifest_sha256':report['provider_manifest_sha256'],'recorder_id':rid},indent=2))

{
  "status": "price_only_forward_raw_plane_qualified_under_split_source",
  "rows": 50,
  "max_prior_error": 5.15000010636868e-07,
  "max_return_error": 9.196203243355683e-10,
  "provider_manifest_sha256": "b7e5b0d58ec8a673f5cbfafd171330f48f846554b17d7ff60ebb4f3a593d1798",
  "recorder_id": "eb4d3106937c4e21b612d83c47ce0d28"
}


## Conclusion

**Price source repaired; SOXX strategy goal unmet.** The isolated existing-source provider has 25 native dates and 50 asset rows. Every new day's raw-share previous close and split-aware return agrees with retained Futu quotes; Qlib readback passes. The 2026-11-05 3:1 split is represented as a share-count event. Volume, full dividend cash, first receipt, broker fills, model decisions and a native account are not qualified by this card. See `conclusion.md`, `summary.json`, and NAS receipt.